# 00 - Preprocessing — SignBridge

This notebook downloads or locates the Kaggle ASL Signs data, converts raw landmark parquet files into model-ready NumPy arrays, performs participant-based train/validation/test splitting, removes labels that do not appear in every split, normalizes features using train-only statistics, and saves all preprocessing artifacts into the organized project folders.

In [ ]:
# ============================================================
# CELL 1 — Notebook Setup and Path Configuration
# ============================================================
# Purpose:
# - Mount Google Drive
# - Define one consistent project folder structure
# - Create data, model, and results folders used across all notebooks

from google.colab import drive

drive.mount('/content/drive')

import os
import sys
import json
import random
import time
import zipfile
from google.colab import userdata
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Change this only if your project folder is somewhere else.
PROJECT_ROOT = '/content/drive/MyDrive/UChicago/Masters/Spring/ADSP 31018/SignBridge'

SRC_DIR = os.path.join(PROJECT_ROOT, 'src')

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
EXTERNAL_DIR = os.path.join(DATA_DIR, 'external')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')

MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
CHECKPOINT_DIR = os.path.join(MODEL_DIR, 'checkpoints')
ONNX_DIR = os.path.join(MODEL_DIR, 'onnx')
TORCHSCRIPT_DIR = os.path.join(MODEL_DIR, 'torchscript')
LOGS_DIR = os.path.join(MODEL_DIR, 'logs')

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
GRAPH_DIR = os.path.join(RESULTS_DIR, 'graphs')
REPORT_DIR = os.path.join(RESULTS_DIR, 'reports')
TABLE_DIR = os.path.join(RESULTS_DIR, 'tables')

# Preprocessing output folder for this notebook
OUTPUT_DIR = PROCESSED_DIR

for folder in [
    SRC_DIR,
    DATA_DIR,
    EXTERNAL_DIR,
    PROCESSED_DIR,
    INPUT_DIR,
    MODEL_DIR,
    CHECKPOINT_DIR,
    ONNX_DIR,
    TORCHSCRIPT_DIR,
    LOGS_DIR,
    RESULTS_DIR,
    GRAPH_DIR,
    REPORT_DIR,
    TABLE_DIR,
]:
    os.makedirs(folder, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Project root:', PROJECT_ROOT)
print('Raw data dir:', RAW_DIR)
print('External data dir:', EXTERNAL_DIR)
print('Processed output dir:', OUTPUT_DIR)
print('Results dir:', RESULTS_DIR)

In [ ]:
# ============================================================
# CELL 2 — Locate ASL Signs Input Data
# ============================================================

import pandas as pd
import numpy as np
import os
import json
import zipfile
from tqdm.notebook import tqdm
from collections import defaultdict
from google.colab import userdata

In [ ]:
# ============================================================
# CELL 3 — Locate ASL Signs Input Data
# ============================================================
# Purpose:
# - Use an already-extracted ASL Signs folder if it exists
# - Otherwise extract asl-signs.zip from data/external
# - Keep all output files inside data/processed/signbridge_all250_clean

# Expected raw Kaggle structure:
#   train.csv
#   train_landmark_files/

PREFERRED_INPUT_DIRS = [
    os.path.join(EXTERNAL_DIR, 'asl-signs'),
    os.path.join(RAW_DIR, 'asl-signs'),
    '/content/asl-signs',
    '/content/asl_signs_data',
]

INPUT_DIR = None

for candidate in PREFERRED_INPUT_DIRS:
    if (
        os.path.exists(os.path.join(candidate, 'train.csv'))
        and os.path.exists(os.path.join(candidate, 'train_landmark_files'))
    ):
        INPUT_DIR = candidate
        print('Using existing dataset:', INPUT_DIR)
        break

if INPUT_DIR is None:
    ZIP_PATH = os.path.join(EXTERNAL_DIR, 'asl-signs.zip')
    INPUT_DIR = os.path.join(EXTERNAL_DIR, 'asl-signs')

    print('No extracted dataset found.')
    print('Looking for local zip:', ZIP_PATH)

    assert os.path.exists(ZIP_PATH), (
        'Could not find asl-signs.zip. '
        'Download it manually from Kaggle and place it in data/external/.'
    )

    os.makedirs(INPUT_DIR, exist_ok=True)

    print('Extracting local Kaggle zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(INPUT_DIR)

print('INPUT_DIR:', INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

print('\nContents of INPUT_DIR:')
for fname in sorted(os.listdir(INPUT_DIR))[:20]:
    fpath = os.path.join(INPUT_DIR, fname)

    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        size_text = f'{size / 1e6:.1f} MB' if size > 1e6 else f'{size / 1024:.1f} KB'
    else:
        size_text = '<DIR>'

    print(f'  {fname:<40} {size_text}')

assert os.path.exists(os.path.join(INPUT_DIR, 'train.csv')), 'train.csv not found.'
assert os.path.exists(os.path.join(INPUT_DIR, 'train_landmark_files')), 'train_landmark_files not found.'

print('\nDataset ready.')

100%|██████████| 37.4G/37.4G [16:26<00:00, 40.7MB/s]

Extracting files...


Input dir:  /root/.cache/kagglehub/competitions/asl-signs
Output dir: /content/drive/Shareddrives/ML2 Final/SignBridge_lawrence/data/raw/arthur/signbridge_all250

Contents of INPUT_DIR:
  sign_to_prediction_index_map.json        3.3 KB
  train.csv                                6.4 MB
  train_landmark_files                     4.0 KB


In [ ]:
# ============================================================
# CELL 4 — Define Vocabulary and Label Mapping
# ============================================================

train = pd.read_csv(f'{INPUT_DIR}/train.csv')
print(f'Total clips in train.csv: {len(train):,}')
print(f'Columns: {list(train.columns)}')
print(f'Participants: {train["participant_id"].nunique()}')
print(f'Signs: {train["sign"].nunique()}')
print()
print('Clips per participant:')
print(train.groupby('participant_id').size().sort_values(ascending=False).to_string())

Total clips in train.csv: 94,477
Columns: ['path', 'participant_id', 'sequence_id', 'sign']
Participants: 21
Signs: 250

Clips per participant:
participant_id
49445    4968
61333    4900
36257    4896
16069    4848
26734    4841
55372    4826
2044     4810
37779    4782
32319    4753
29302    4722
22343    4677
53618    4656
37055    4648
28656    4563
62590    4563
34503    4545
27610    4275
25571    3865
18796    3502
4718     3499
30680    3338


## Config

In [ ]:
# ============================================================
# CELL 5 — Create Participant-Based Train Val Test Split
# ============================================================

MAX_SEQ_LEN  = 96
MIN_FRAMES   = 10

# ── Relaxed skip threshold ─────────────────────────────────────
# V1 used 0.80 on right hand only → dropped 57.6% of clips
# V2: drop only if BOTH hands missing in >50% of frames
BOTH_HAND_MISSING_THRESH = 0.50

# ── Landmark selection (Hoyeol Sohn 1st place) ────────────────
NOSE = [1, 2, 98, 327]
LIP  = [
    0, 61, 185, 40, 39, 37, 267, 269, 270, 409,
    291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
    95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
]
REYE = [
    33, 7, 163, 144, 145, 153, 154, 155, 133,
    246, 161, 160, 159, 158, 157, 173,
]
LEYE = [
    263, 249, 390, 373, 374, 380, 381, 382, 362,
    466, 388, 387, 386, 385, 384, 398,
]
KEEP = {
    'face'      : LIP + NOSE + REYE + LEYE,
    'left_hand' : list(range(21)),
    'right_hand': list(range(21)),
}

# ── Build column template ──────────────────────────────────────
EXPECTED_COLS = []
for coord in ['x', 'y', 'z']:
    for ltype in sorted(KEEP.keys()):
        for idx in sorted(KEEP[ltype]):
            EXPECTED_COLS.append(f'{coord}_{ltype}_{idx}')

N_FEATURES = len(EXPECTED_COLS)

# Precompute hand column masks
LH_MASK = np.array(['left_hand'  in c for c in EXPECTED_COLS])
RH_MASK = np.array(['right_hand' in c for c in EXPECTED_COLS])

print(f'Landmarks      : 118')
print(f'Features/frame : {N_FEATURES} position + {N_FEATURES} velocity = {N_FEATURES*2} total')
print(f'Sequence length: {MAX_SEQ_LEN} frames')

# ── Vocabulary ─────────────────────────────────────────────────
VOCAB         = sorted(train['sign'].unique())
word_to_label = {w: i for i, w in enumerate(VOCAB)}
label_to_word = {i: w for w, i in word_to_label.items()}
print(f'\nTotal signs: {len(VOCAB)}')
print(f'Total clips: {len(train):,}')

Landmarks      : 118
Features/frame : 354 position + 354 velocity = 708 total
Sequence length: 96 frames

Total signs: 250
Total clips: 94,477


## Participant Split

**V1 problem:** used `participants[-1]` and `participants[-2]` — arbitrary array order that happened to pick participant 30680 for test, who signs differently from training signers.

**V2 fix:** score each participant by class coverage (how many of the 250 signs they have clips for). Pick participants with the *best* coverage for val/test so evaluation is meaningful across as many classes as possible. If enough participants exist, assign 2+ per val/test split.

In [ ]:
# ============================================================
# CELL 6 — Define Landmark Columns and Constants
# ============================================================

# ── Analyse participant coverage ───────────────────────────────
participant_coverage = (
    train.groupby('participant_id')['sign']
    .nunique()
    .sort_values(ascending=False)
    .rename('n_unique_signs')
)
participant_clips = (
    train.groupby('participant_id')
    .size()
    .rename('n_clips')
)
participant_info = pd.concat([participant_coverage, participant_clips], axis=1)
print('Participant coverage summary:')
print(participant_info.to_string())
print(f'\nTotal participants: {len(participant_info)}')

Participant coverage summary:
                n_unique_signs  n_clips
participant_id                         
2044                       250     4810
4718                       250     3499
16069                      250     4848
18796                      250     3502
22343                      250     4677
26734                      250     4841
27610                      250     4275
28656                      250     4563
29302                      250     4722
34503                      250     4545
32319                      250     4753
37055                      250     4648
36257                      250     4896
53618                      250     4656
55372                      250     4826
37779                      250     4782
49445                      250     4968
61333                      250     4900
62590                      250     4563
25571                      242     3865
30680                      238     3338

Total participants: 21


In [ ]:
# ============================================================
# CELL 7 — Define Landmark Preprocessing Functions
# ============================================================

# ── Assign val/test participants ───────────────────────────────
# Strategy: sort by coverage descending, assign top participants
# to val and test so evaluation covers as many classes as possible.
# Keep enough in train to have learning signal.

all_participants = participant_info.index.tolist()  # sorted by coverage desc
n_total          = len(all_participants)

# Use ~15% for test, ~15% for val — minimum 1 each
n_test = max(1, int(round(n_total * 0.15)))
n_val  = max(1, int(round(n_total * 0.15)))

# Pick from the best-coverage participants so splits have fewer missing classes
# Interleave: take every other from top so both val and test get good coverage
top_participants = all_participants[:n_test + n_val]
test_participants = set(top_participants[0::2][:n_test])
val_participants  = set(top_participants[1::2][:n_val])

# Ensure no overlap (shouldn't happen but guard anyway)
assert len(test_participants & val_participants) == 0
train_participants = set(all_participants) - test_participants - val_participants

print(f'Train participants ({len(train_participants)}): {sorted(train_participants)}')
print(f'Val   participants ({len(val_participants)}):   {sorted(val_participants)}')
print(f'Test  participants ({len(test_participants)}):  {sorted(test_participants)}')

# Preview coverage
for name, pids in [('Val', val_participants), ('Test', test_participants)]:
    mask     = train['participant_id'].isin(pids)
    coverage = train[mask]['sign'].nunique()
    clips    = mask.sum()
    print(f'\n{name}: {clips} clips, {coverage}/250 classes covered')

Train participants (15): [25571, 27610, 28656, 29302, 30680, 32319, 34503, 36257, 37055, 37779, 49445, 53618, 55372, 61333, 62590]
Val   participants (3):   [4718, 18796, 26734]
Test  participants (3):  [2044, 16069, 22343]

Val: 11842 clips, 250/250 classes covered

Test: 14335 clips, 250/250 classes covered


## Landmark Imputation

**V1 problem:** dropped entire clips if right hand was missing >80% of frames. Left-dominant signs and fast-motion signs (down, drop, go) were disproportionately skipped — 57.6% of all data.

**V2 fix:** interpolate missing hand frames between detected neighbours. Only drop if both hands are simultaneously missing in >50% of frames (truly undetectable clip).

In [ ]:
# ============================================================
# CELL 8 — Process All Clips into Sequences
# ============================================================

def impute_hand(seq, hand_mask):
    """
    Interpolate missing hand frames in-place.
    A frame is 'missing' if all hand landmark values are exactly 0.

    seq       : (n_frames, n_features) float32 array
    hand_mask : boolean array of length n_features, True for this hand's columns
    Returns   : (seq_modified, fraction_missing_before_imputation)
    """
    hand_cols  = np.where(hand_mask)[0]
    n_frames   = len(seq)

    # A frame is missing if all landmark values for this hand are 0
    missing = np.all(seq[:, hand_cols] == 0, axis=1)  # (n_frames,)
    frac_missing = missing.mean()

    if frac_missing == 0:
        return seq, 0.0  # nothing to do

    if frac_missing == 1.0:
        return seq, 1.0  # entire hand never detected — leave as zeros

    detected = np.where(~missing)[0]

    for i in np.where(missing)[0]:
        before = detected[detected < i]
        after  = detected[detected > i]

        if len(before) > 0 and len(after) > 0:
            # Linear interpolation between nearest detected neighbours
            b, a  = before[-1], after[0]
            alpha = (i - b) / (a - b)
            seq[i, hand_cols] = (
                (1 - alpha) * seq[b, hand_cols] +
                alpha       * seq[a, hand_cols]
            )
        elif len(before) > 0:
            seq[i, hand_cols] = seq[before[-1], hand_cols]  # forward fill
        elif len(after) > 0:
            seq[i, hand_cols] = seq[after[0],   hand_cols]  # backward fill

    return seq, frac_missing


def process_clip(path):
    """
    Process a single .parquet clip file.
    Returns (seq, actual_len, skip_reason) where seq is None if clip is unusable.
    """
    try:
        df = pd.read_parquet(path)
    except Exception:
        return None, None, 'read_error'

    # ── Select landmarks ──────────────────────────────────────
    mask = pd.Series(False, index=df.index)
    for ltype, indices in KEEP.items():
        mask |= ((df['type'] == ltype) &
                 (df['landmark_index'].isin(indices)))
    df = df[mask]
    if len(df) == 0:
        return None, None, 'no_landmarks'

    # ── Pivot long → wide ──────────────────────────────────────
    try:
        wide = df.pivot_table(
            index='frame',
            columns=['type', 'landmark_index'],
            values=['x', 'y', 'z'],
            aggfunc='first'
        )
        wide.columns = [f'{c}_{t}_{i}' for c, t, i in wide.columns]
        wide = wide.sort_index().reset_index(drop=True)
    except Exception:
        return None, None, 'pivot_error'

    # Force identical columns
    for col in EXPECTED_COLS:
        if col not in wide.columns:
            wide[col] = 0.0
    wide = wide[EXPECTED_COLS]

    actual_len = len(wide)
    if actual_len < MIN_FRAMES:
        return None, None, 'too_short'

    seq = wide.fillna(0).values.astype(np.float32)

    # ── Impute missing hand frames ────────────────────────────
    seq, lh_missing_frac = impute_hand(seq, LH_MASK)
    seq, rh_missing_frac = impute_hand(seq, RH_MASK)

    # Drop only if BOTH hands are missing in >50% of frames
    lh_missing_frames = np.all(seq[:, LH_MASK] == 0, axis=1)
    rh_missing_frames = np.all(seq[:, RH_MASK] == 0, axis=1)
    both_missing_frac = np.mean(lh_missing_frames & rh_missing_frames)

    if both_missing_frac > BOTH_HAND_MISSING_THRESH:
        return None, None, 'both_hands_missing'

    # ── Normalize: subtract right wrist, divide by hand size ──
    try:
        wx = EXPECTED_COLS.index('x_right_hand_0')
        wy = EXPECTED_COLS.index('y_right_hand_0')
        wz = EXPECTED_COLS.index('z_right_hand_0')
        wrist_x = seq[:, wx:wx+1]
        wrist_y = seq[:, wy:wy+1]
        wrist_z = seq[:, wz:wz+1]
        for i, c in enumerate(EXPECTED_COLS):
            if 'right_hand' in c:
                if   c[0] == 'x': seq[:, i] -= wrist_x[:, 0]
                elif c[0] == 'y': seq[:, i] -= wrist_y[:, 0]
                elif c[0] == 'z': seq[:, i] -= wrist_z[:, 0]
    except ValueError:
        pass

    try:
        mx    = EXPECTED_COLS.index('x_right_hand_12')
        my    = EXPECTED_COLS.index('y_right_hand_12')
        mz    = EXPECTED_COLS.index('z_right_hand_12')
        scale = np.linalg.norm(seq[:, [mx, my, mz]], axis=1, keepdims=True)
        scale = np.where(scale < 1e-6, 1.0, scale)
        for i, c in enumerate(EXPECTED_COLS):
            if 'right_hand' in c:
                seq[:, i] /= scale[:, 0]
    except ValueError:
        pass

    # ── Pad or centre-crop to MAX_SEQ_LEN ─────────────────────
    if actual_len > MAX_SEQ_LEN:
        start      = (actual_len - MAX_SEQ_LEN) // 2
        seq        = seq[start : start + MAX_SEQ_LEN]
        actual_len = MAX_SEQ_LEN
    elif actual_len < MAX_SEQ_LEN:
        pad = np.zeros((MAX_SEQ_LEN - actual_len, seq.shape[1]), dtype=np.float32)
        seq = np.concatenate([seq, pad], axis=0)

    # ── Append velocity ───────────────────────────────────────
    velocity = np.diff(seq, axis=0, prepend=seq[:1]).astype(np.float32)
    seq = np.concatenate([seq, velocity], axis=1)

    return seq, actual_len, None  # None = no skip reason

## Process All Clips

In [ ]:
# ============================================================
# CELL 9 — Convert Lists to NumPy Arrays
# ============================================================

X_train, y_train, len_train = [], [], []
X_val,   y_val,   len_val   = [], [], []
X_test,  y_test,  len_test  = [], [], []

skipped    = 0
processed  = 0
skip_reasons = defaultdict(int)
sign_stats   = {w: {'ok': 0, 'skip': 0} for w in VOCAB}

for _, row in tqdm(train.iterrows(), total=len(train), desc='Processing clips'):

    path = (
        f'{INPUT_DIR}/train_landmark_files/'
        f'{row["participant_id"]}/'
        f'{row["sequence_id"]}.parquet'
    )

    seq, actual_len, reason = process_clip(path)
    word = row['sign']

    if seq is None:
        skipped += 1
        skip_reasons[reason] += 1
        sign_stats[word]['skip'] += 1
        continue

    label = word_to_label[word]
    pid   = row['participant_id']

    if pid in test_participants:
        X_test.append(seq);  y_test.append(label);  len_test.append(actual_len)
    elif pid in val_participants:
        X_val.append(seq);   y_val.append(label);   len_val.append(actual_len)
    else:
        X_train.append(seq); y_train. append(label); len_train.append(actual_len)

    sign_stats[word]['ok'] += 1
    processed += 1

print(f'\nProcessed : {processed:,}')
print(f'Skipped   : {skipped:,}  ({skipped/(processed+skipped)*100:.1f}%)')
print(f'\nSkip reasons:')
for reason, count in sorted(skip_reasons.items(), key=lambda x: -x[1]):
    print(f'  {reason:<25} {count:>6}')

Processing clips:   0%|          | 0/94477 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
/tmp/ipykernel_668/1553056008.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wide[col] = 0.0
/tmp/ipykernel_668/1553056008.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wide[col] = 0.0
/tmp/ipykernel_668/1553056008.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, u


Processed : 76,759
Skipped   : 17,718  (18.8%)

Skip reasons:
  too_short                  17718


In [ ]:
# ============================================================
# CELL 10 — Normalize Features
# ============================================================

# ── Convert to numpy ───────────────────────────────────────────
X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.int32)
l_train = np.array(len_train, dtype=np.int32)

X_val   = np.array(X_val,   dtype=np.float32)
y_val   = np.array(y_val,   dtype=np.int32)
l_val   = np.array(len_val, dtype=np.int32)

X_test  = np.array(X_test,  dtype=np.float32)
y_test  = np.array(y_test,  dtype=np.int32)
l_test  = np.array(len_test, dtype=np.int32)

print('Shapes:')
print(f'  X_train : {X_train.shape}')
print(f'  X_val   : {X_val.shape}')
print(f'  X_test  : {X_test.shape}')

Shapes:
  X_train : (55642, 96, 708)
  X_val   : (9291, 96, 708)
  X_test  : (11826, 96, 708)


In [ ]:
# ============================================================
# CELL 11 — Save Processed Arrays and Metadata
# ============================================================

# ── Coverage check ─────────────────────────────────────────────
train_counts = np.bincount(y_train, minlength=len(VOCAB))
val_counts   = np.bincount(y_val,   minlength=len(VOCAB))
test_counts  = np.bincount(y_test,  minlength=len(VOCAB))

print(f'Classes with 0 train samples: {(train_counts == 0).sum()}')
print(f'Classes with 0 val   samples: {(val_counts   == 0).sum()}')
print(f'Classes with 0 test  samples: {(test_counts  == 0).sum()}')
print(f'\nMean clips per class:')
print(f'  Train: {train_counts[train_counts>0].mean():.1f}')
print(f'  Val:   {val_counts[val_counts>0].mean():.1f}')
print(f'  Test:  {test_counts[test_counts>0].mean():.1f}')

# Per-split coverage correlation (should be close to 1.0)
corr_tv = np.corrcoef(train_counts, val_counts)[0, 1]
corr_tt = np.corrcoef(train_counts, test_counts)[0, 1]
print(f'\nPer-class count correlation (train vs val):  {corr_tv:.4f}')
print(f'Per-class count correlation (train vs test): {corr_tt:.4f}')

if (val_counts == 0).sum() > 10 or (test_counts == 0).sum() > 10:
    print('\n⚠  Still many missing classes. Consider increasing n_val/n_test participants above.')
else:
    print('\n✓  Coverage looks good.')

Classes with 0 train samples: 0
Classes with 0 val   samples: 0
Classes with 0 test  samples: 0

Mean clips per class:
  Train: 222.6
  Val:   37.2
  Test:  47.3

Per-class count correlation (train vs val):  0.7143
Per-class count correlation (train vs test): 0.7952

✓  Coverage looks good.


## Normalisation

Fit mean/std on train only, apply to all splits. **Save the normaliser** so inference can apply the same transform at runtime.

In [ ]:
# ============================================================
# CELL 12 — Verify Saved Output Files
# ============================================================

# ── Fit on train only ──────────────────────────────────────────
# Reshape to (n_samples * seq_len, n_features) for per-feature stats
n_train, seq_len, n_feat = X_train.shape
X_flat = X_train.reshape(-1, n_feat)

feat_mean = X_flat.mean(axis=0).astype(np.float32)
feat_std  = X_flat.std(axis=0).astype(np.float32)
feat_std  = np.where(feat_std < 1e-6, 1.0, feat_std)  # avoid div-by-zero

# ── Apply to all splits ────────────────────────────────────────
X_train_norm = ((X_train - feat_mean) / feat_std).astype(np.float32)
X_val_norm   = ((X_val   - feat_mean) / feat_std).astype(np.float32)
X_test_norm  = ((X_test  - feat_mean) / feat_std).astype(np.float32)

print(f'Train mean (sample): {X_train_norm.mean():.6f}  (should be ~0)')
print(f'Train std  (sample): {X_train_norm.std():.6f}   (should be ~1)')
print(f'Val   mean (sample): {X_val_norm.mean():.6f}')
print(f'Test  mean (sample): {X_test_norm.mean():.6f}')

Train mean (sample): -0.000011  (should be ~0)
Train std  (sample): 0.998157   (should be ~1)
Val   mean (sample): 0.019822
Test  mean (sample): -0.026745


## Save

In [ ]:
# ============================================================
# CELL 13 — Dataset Statistics Summary
# ============================================================

print('Saving...')

# Raw (un-normalised) arrays
np.savez_compressed(f'{OUTPUT_DIR}/X_train.npz', data=X_train, lengths=l_train)
np.savez_compressed(f'{OUTPUT_DIR}/X_val.npz',   data=X_val,   lengths=l_val)
np.savez_compressed(f'{OUTPUT_DIR}/X_test.npz',  data=X_test,  lengths=l_test)

# Normalised arrays (what the model consumes)
np.savez_compressed(f'{OUTPUT_DIR}/X_train_norm.npz', data=X_train_norm)
np.savez_compressed(f'{OUTPUT_DIR}/X_val_norm.npz',   data=X_val_norm)
np.savez_compressed(f'{OUTPUT_DIR}/X_test_norm.npz',  data=X_test_norm)

# Labels and lengths
np.save(f'{OUTPUT_DIR}/y_train.npy',       y_train)
np.save(f'{OUTPUT_DIR}/y_val.npy',         y_val)
np.save(f'{OUTPUT_DIR}/y_test.npy',        y_test)
np.save(f'{OUTPUT_DIR}/lengths_train.npy', l_train)
np.save(f'{OUTPUT_DIR}/lengths_val.npy',   l_val)
np.save(f'{OUTPUT_DIR}/lengths_test.npy',  l_test)

# Normaliser — REQUIRED for inference
np.save(f'{OUTPUT_DIR}/feat_mean.npy', feat_mean)
np.save(f'{OUTPUT_DIR}/feat_std.npy',  feat_std)

# Metadata
metadata = {
    'vocab'             : VOCAB,
    'word_to_label'     : word_to_label,
    'label_to_word'     : {str(k): v for k, v in label_to_word.items()},
    'max_seq_len'       : MAX_SEQ_LEN,
    'min_frames'        : MIN_FRAMES,
    'nan_thresh'        : BOTH_HAND_MISSING_THRESH,
    'n_features'        : int(X_train.shape[2]),
    'n_features_pos'    : N_FEATURES,
    'n_features_vel'    : N_FEATURES,
    'n_landmarks'       : 118,
    'n_words'           : len(VOCAB),
    'n_train'           : int(len(X_train)),
    'n_val'             : int(len(X_val)),
    'n_test'            : int(len(X_test)),
    'n_skipped'         : skipped,
    'skip_reasons'      : dict(skip_reasons),
    'val_participants'  : sorted(val_participants),
    'test_participants' : sorted(test_participants),
    'train_participants': sorted(train_participants),
    'landmark_source'   : 'Hoyeol Sohn 1st place asl-signs 2023',
    'sign_stats'        : sign_stats,
    'preprocessing_version': 'v2',
}
with open(f'{OUTPUT_DIR}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('\nSaved files:')
print(f'{"File":<30} {"Size":>10}')
print('─' * 42)
total_bytes = 0
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = f'{OUTPUT_DIR}/{fname}'
    size  = os.path.getsize(fpath)
    total_bytes += size
    size_str = f'{size/1e6:.1f} MB' if size > 1e6 else f'{size/1024:.1f} KB'
    print(f'{fname:<30} {size_str:>10}')
print('─' * 42)
print(f'{"TOTAL":<30} {total_bytes/1e6:>9.1f} MB')

Saving...

Saved files:
File                                 Size
──────────────────────────────────────────
X_test.npz                       865.1 MB
X_test_norm.npz                  915.6 MB
X_train.npz                     4484.7 MB
X_train_norm.npz                4733.7 MB
X_val.npz                        821.5 MB
X_val_norm.npz                   864.7 MB
feat_mean.npy                      2.9 KB
feat_std.npy                       2.9 KB
lengths_test.npy                  46.3 KB
lengths_train.npy                217.5 KB
lengths_val.npy                   36.4 KB
metadata.json                     26.6 KB
y_test.npy                        46.3 KB
y_train.npy                      217.5 KB
y_val.npy                         36.4 KB
──────────────────────────────────────────
TOTAL                            12686.0 MB


In [ ]:
# ============================================================
# CELL 14 — Quick Sample Inspection
# ============================================================

# ── Final summary ──────────────────────────────────────────────
skip_rate = skipped / (processed + skipped) * 100

print(f'''
╔══════════════════════════════════════════════════╗
║        PREPROCESSING V2 — FINAL SUMMARY         ║
╠══════════════════════════════════════════════════╣
║  Signs processed  : {len(VOCAB):<5}                       ║
║  Train clips      : {len(X_train):<6,}                      ║
║  Val clips        : {len(X_val):<6,}                      ║
║  Test clips       : {len(X_test):<6,}                      ║
║  Skipped          : {skipped:<6,}  ({skip_rate:.1f}%)             ║
╠══════════════════════════════════════════════════╣
║  Shape: {str(X_train.shape):<39} ║
╠══════════════════════════════════════════════════╣
║  Val  participants : {str(sorted(val_participants)):<28} ║
║  Test participants : {str(sorted(test_participants)):<28} ║
╠══════════════════════════════════════════════════╣
║  Val  classes covered : {(val_counts  > 0).sum():<3}/250                 ║
║  Test classes covered : {(test_counts > 0).sum():<3}/250                 ║
╚══════════════════════════════════════════════════╝
''')

# Signs still with low coverage
low = [(w, s['ok'], s['skip']) for w, s in sign_stats.items() if s['ok'] < 5]
if low:
    print(f'Signs with <5 usable clips ({len(low)}):')
    for word, ok, skip in sorted(low):
        print(f'  {word:<25}  ok={ok}  skip={skip}')
else:
    print('✓  All signs have at least 5 usable clips.')


╔══════════════════════════════════════════════════╗
║        PREPROCESSING V2 — FINAL SUMMARY         ║
╠══════════════════════════════════════════════════╣
║  Signs processed  : 250                         ║
║  Train clips      : 55,642                      ║
║  Val clips        : 9,291                       ║
║  Test clips       : 11,826                      ║
║  Skipped          : 17,718  (18.8%)             ║
╠══════════════════════════════════════════════════╣
║  Shape: (55642, 96, 708)                        ║
╠══════════════════════════════════════════════════╣
║  Val  participants : [4718, 18796, 26734]         ║
║  Test participants : [2044, 16069, 22343]         ║
╠══════════════════════════════════════════════════╣
║  Val  classes covered : 250/250                 ║
║  Test classes covered : 250/250                 ║
╚══════════════════════════════════════════════════╝

✓  All signs have at least 5 usable clips.
